<a href="https://colab.research.google.com/github/GitSarraa/TP_IA_Project/blob/main/image_traitement/Prediction_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount ('/content/gdrive')

Mounted at /content/gdrive


In [2]:
!pip install ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 11.2 MB/s eta 0:00:00


In [9]:
from ultralytics import YOLO  #pour importer notre modèle YOLO
import cv2 # pour le traitement de nos images
import numpy as np # pour gérer des fonctions et des tableaux
import os #pour le traitement de fichiers et dossiers
import re
from pathlib import Path #pour gérer les chemins
from datetime import datetime #pour extraire la date et l'heure de prise des images
from PIL import Image #accéder aux data brutes de l'images
from PIL.ExifTags import TAGS, IFD #traduire les codes numériques en texte lisible


# 1. Fonction CLAHE pour le prétraitement

def apply_clahe(img, clip_limit=2.0):
    """CLAHE sur le canal L de l'espace LAB."""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    l_eq = clahe.apply(l)
    lab_eq = cv2.merge((l_eq, a, b))
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)



# 2. Fonction orthorectification (à partir du fichier GRP.dat)

def read_grp_dat(path):
    """
    Lit un fichier GRP.dat produit par Fudaa-LSPIV.
    Format :
        Ligne 1 : "GRP V2.0 <largeur_image> <hauteur_image>(4000*3000)"
        Ligne 2 : nombre de points (4)
        Ligne 3 : entetes "X Y Z i j"
        Lignes suivantes : donnees
    Retourne :
        pts_image : np.ndarray (N,2) float32 e-> (i, j) en pixels
        pts_reel  : np.ndarray (N,2) float32 -> (X, Y) en metres
        img_size  : (largeur, hauteur) en pixels
    """
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]

    header = lines[0].split()
    img_w, img_h = int(header[2]), int(header[3])

    n_points = int(lines[1])
    data_lines = lines[3 : 3 + n_points]

    pts_image, pts_reel = [], []
    for line in data_lines:
        X, Y, Z, i, j = map(float, line.split())
        pts_reel.append([X, Y])
        pts_image.append([i, j])

    return (
        np.array(pts_image, dtype=np.float32),
        np.array(pts_reel, dtype=np.float32),
        (img_w, img_h),
    )


def compute_homography(grp_path):
    """Calcule H (image -> reel, en metres) à partir d'un GRP.dat à 4 points."""
    pts_image, pts_reel, img_size = read_grp_dat(grp_path)
    if len(pts_image) != 4:
        raise ValueError(
            f"getPerspectiveTransform necessite exactement 4 points, "
            f"trouve {len(pts_image)} dans {grp_path}."
        )
    H = cv2.getPerspectiveTransform(pts_image, pts_reel)
    return H, img_size


def contour_to_real_area(contour_px, H):
    """
    Transforme un contour (polygone en pixels) vers l'espace reel et calcule son aire.
    contour_px : np.ndarray (N,2), coordonnees (i,j) en pixels (ex: results[0].masks.xy[k])
    H          : matrice d'homographie 3x3 (image -> reel, metres)
    Retourne : (contour_reel en m, aire_m2)
    """
    pts = np.asarray(contour_px, dtype=np.float32).reshape(-1, 1, 2)
    pts_reel = cv2.perspectiveTransform(pts, H).reshape(-1, 2)

    x, y = pts_reel[:, 0], pts_reel[:, 1]
    aire_m2 = 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
    return pts_reel, aire_m2


# 2a. LECTURE DE LA DATE/HEURE DE PRISE DE VUE

def get_capture_datetime(img_path):
    """
    Recupere la date/heure de prise de vue.
    1) Essaie l'EXIF DateTimeOriginal (le plus fiable, ecrit par l'appareil photo).
    2) Si absent, essaie d'extraire une date/heure du nom de fichier
       (utile pour les cameras timelapse qui nomment les fichiers avec un horodatage).
    3) Sinon, retourne (None, None).

    Retourne : (date_str "YYYY-MM-DD", heure_str "HH:MM:SS") ou (None, None)
    """
    # --- 1) EXIF ---
    try:
        img = Image.open(img_path)
        exif = img.getexif()
        if exif:
            # Tag "DateTime" (modification), parfois present a la racine
            for tag_id, value in exif.items():
                tag = TAGS.get(tag_id, tag_id)
                if tag == "DateTime":
                    dt = datetime.strptime(value, "%Y:%m:%d %H:%M:%S")
                    return dt.strftime("%Y-%m-%d"), dt.strftime("%H:%M:%S")
            # Tag "DateTimeOriginal" (prise de vue reelle), vit dans le sous-IFD Exif
            try:
                exif_ifd = exif.get_ifd(IFD.Exif)
                for tag_id, value in exif_ifd.items():
                    tag = TAGS.get(tag_id, tag_id)
                    if tag == "DateTimeOriginal":
                        dt = datetime.strptime(value, "%Y:%m:%d %H:%M:%S")
                        return dt.strftime("%Y-%m-%d"), dt.strftime("%H:%M:%S")
            except Exception:
                pass
    except Exception:
        pass

    # --- 2) Repli : essayer de parser le nom de fichier ---
    # Adapter ce pattern au format reel de tes noms de fichiers timelapse si besoin.
    # Exemple gere ici : "20240815_143000.jpg" ou "2024-08-15_14-30-00.jpg"
    name = Path(img_path).stem
    match = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})[_T-]?(\d{2})[-:]?(\d{2})[-:]?(\d{2})", name)
    if match:
        y, mo, d, h_, mi, s = match.groups()
        try:
            dt = datetime(int(y), int(mo), int(d), int(h_), int(mi), int(s))
            return dt.strftime("%Y-%m-%d"), dt.strftime("%H:%M:%S")
        except ValueError:
            pass

    # --- 3) Rien trouve ---
    return None, None



# 3. Configuration des chamins

MODEL_PATH = '/content/gdrive/MyDrive/SARAH/train_yolo11m_500_1classe/weights/best.pt'
SOURCE_FOLDER = '/content/gdrive/MyDrive/SARAH/Bains_Esparre_atester'
OUTPUT_FOLDER = '/content/gdrive/MyDrive/SARAH/train_yolo11m_500_1classe/predictions/predisction&summary70%'

# Fichier de points de reference Fudaa-LSPIV (un par site/camera fixe)
GRP_PATH = '/content/gdrive/MyDrive/SARAH/GRP_BainsEsparre.dat'

# Chargement du modèle
model = YOLO(MODEL_PATH)

# Chargement de l'homographie (une seule fois, valable pour toute la sequence
# de cette camera fixe)
H, (grp_img_w, grp_img_h) = compute_homography(GRP_PATH)
print(f" Homographie chargee depuis {GRP_PATH} (image de reference {grp_img_w}x{grp_img_h})")

# Création des dossiers de sortie
side_by_side_dir = os.path.join(OUTPUT_FOLDER, 'side_by_side')
label_dir = os.path.join(OUTPUT_FOLDER, 'labels')
surfaces_dir = os.path.join(OUTPUT_FOLDER, 'surfaces')
os.makedirs(side_by_side_dir, exist_ok=True)
os.makedirs(label_dir, exist_ok=True)
os.makedirs(surfaces_dir, exist_ok=True)

# Récupération de toutes les images
image_extensions = {'.jpg', '.jpeg', '.png'}
image_paths = [p for p in Path(SOURCE_FOLDER).iterdir() if p.suffix.lower() in image_extensions]

print(f" {len(image_paths)} images trouvées dans {SOURCE_FOLDER}")

# Fichier recapitulatif de toutes les surfaces (une ligne par image)
summary_path = os.path.join(surfaces_dir, "surfaces_summary.csv")
summary_rows = ["image,date,heure,surface_m2"]


# 4. Boucle sur chaque image

for img_path in image_paths:
    print(f"\n Traitement de : {img_path.name}")

    # 4a. Chargement de l'image originale
    orig_img = cv2.imread(str(img_path))
    if orig_img is None:
        print(f"     Impossible de lire {img_path.name}")
        continue
    h, w = orig_img.shape[:2]

    # Attention : GRP.dat a ete construit sur une image de taille (grp_img_w, grp_img_h).
    # Si les images a predire n'ont pas exactement cette taille, l'homographie ne
    # correspond plus pixel a pixel. On avertit plutot que d'echouer silencieusement.
    if (w, h) != (grp_img_w, grp_img_h):
        print(f"    Taille image ({w}x{h}) != taille GRP.dat ({grp_img_w}x{grp_img_h}) "
              f"-> verifier la coherence avant d'utiliser les surfaces calculees.")

    # 4b. PRÉTRAITEMENT : Application de CLAHE
    processed_img = apply_clahe(orig_img)

    # 4c. Inférence sur l'image prétraitée
    results = model.predict(
        source=processed_img,
        conf=0.35,
        iou=0.45,
        imgsz=1280,
        device=0,
        verbose=False
    )

    # 4d. Récupération de l'image avec les boîtes/masques dessinés par YOLO
    pred_img_with_boxes = results[0].plot()  # Image en BGR

    # Redimensionnement (pour que les 2 images aient la même hauteur)
    if pred_img_with_boxes.shape[0] != h:
        pred_img_with_boxes = cv2.resize(pred_img_with_boxes, (w, h))

    # 4e. Concaténation côte à côte (originale à gauche, prédite à droite)
    combined = np.hstack((orig_img, pred_img_with_boxes))
    side_path = os.path.join(side_by_side_dir, img_path.name)
    cv2.imwrite(side_path, combined)
    print(f"   ✅ Comparaison sauvegardée : {side_path}")

    # 4f. Sauvegarde des labels .txt (segmentation ou boîtes) + calcul des surfaces
    label_path = os.path.join(label_dir, f"{img_path.stem}.txt")
    n_instances = 0
    surface_totale_m2 = 0.0

    with open(label_path, 'w') as f:
        # Cas où le modèle prédit des masques (segmentation)
        if results[0].masks is not None:
            masks = results[0].masks.xy  # Liste de polygones en pixels
            boxes = results[0].boxes
            for i, polygon in enumerate(masks):
                cls_id = int(boxes.cls[i].item())

                # --- Labels YOLO normalises (comme avant) ---
                flat_coords = polygon.flatten().tolist()
                norm_coords = []
                for idx, val in enumerate(flat_coords):
                    if idx % 2 == 0:
                        norm_coords.append(f"{val / w:.6f}")
                    else:
                        norm_coords.append(f"{val / h:.6f}")
                f.write(f"{cls_id} " + " ".join(norm_coords) + "\n")

                # --- Orthorectification directe du contour + aire reelle ---
                _, aire_m2 = contour_to_real_area(polygon, H)
                surface_totale_m2 += aire_m2
                n_instances += 1

        # Cas où le modèle prédit seulement des boîtes (detection) — pas d'aire possible
        elif results[0].boxes is not None:
            for box in results[0].boxes:
                cls_id = int(box.cls.item())
                x, y, w_box, h_box = box.xywhn[0].tolist()
                f.write(f"{cls_id} {x:.6f} {y:.6f} {w_box:.6f} {h_box:.6f}\n")

    print(f"    Labels sauvegardés : {label_path}")
    print(f"    Surface sable détectée : {surface_totale_m2:.3f} m² ({n_instances} instance(s))")

    # Date/heure de prise de vue (EXIF, ou repli sur le nom de fichier)
    date_str, heure_str = get_capture_datetime(img_path)
    if date_str is None:
        print(f"     Date/heure introuvable pour {img_path.name} (ni EXIF, ni nom de fichier)")
        date_str, heure_str = "", ""

    summary_rows.append(f"{img_path.name},{date_str},{heure_str},{surface_totale_m2:.4f}")

# Ecriture du recapitulatif CSV
with open(summary_path, "w") as f:
    f.write("\n".join(summary_rows) + "\n")


# 5. Affichage

print("\n" + "="*50)
print(" Inférence terminée orthorectification + surfaces m² !")
print(f" Résultats dans : {OUTPUT_FOLDER}")
print(f" Récapitulatif des surfaces : {summary_path}")
print("="*50)

📐 Homographie chargee depuis /content/gdrive/MyDrive/SARAH/GRP_BainsEsparre.dat (image de reference 4000x3000)
📸 61 images trouvées dans /content/gdrive/MyDrive/SARAH/Bains_Esparre_atester

🔄 Traitement de : IMAG0630.JPG
   ✅ Comparaison sauvegardée : /content/gdrive/MyDrive/SARAH/train_yolo11m_500_1classe/predictions/predisction&summary70%/side_by_side/IMAG0630.JPG
   ✅ Labels sauvegardés : /content/gdrive/MyDrive/SARAH/train_yolo11m_500_1classe/predictions/predisction&summary70%/labels/IMAG0630.txt
   📏 Surface sable détectée : 1.644 m² (2 instance(s))

🔄 Traitement de : IMAG0631.JPG
   ✅ Comparaison sauvegardée : /content/gdrive/MyDrive/SARAH/train_yolo11m_500_1classe/predictions/predisction&summary70%/side_by_side/IMAG0631.JPG
   ✅ Labels sauvegardés : /content/gdrive/MyDrive/SARAH/train_yolo11m_500_1classe/predictions/predisction&summary70%/labels/IMAG0631.txt
   📏 Surface sable détectée : 2.762 m² (3 instance(s))

🔄 Traitement de : IMAG0632.JPG
   ✅ Comparaison sauvegardée : /con